# Wangara Diurnal Cycle: Description


**Reference**

Basu, S., Vinuesa, J.-F., and Swift, A. (2008),
Dynamic LES modeling of a diurnal cycle.
*Journal of Applied Meteorology and Climatology*, 47(4), 1156-1174.

Hicks, B. B. (1981),
An analysis of Wangara micrometeorology: Surface stress, sensible heat, evaporation, and dewfall.
NOAA Technical Memorandum ERL ARL-104.

**Case Description**

The Wangara diurnal-cycle case simulates a full 24-hour period from 0900 LST 16 August 1967
(Day 33) to 0900 LST 17 August 1967 (Day 34) over the Wangara experimental site in Australia.
The simulation captures the daytime growth of a convective boundary layer followed by the
nocturnal collapse into a stable boundary layer. The surface is driven by the observed
screen-level temperature at 1.2 m, and the geostrophic wind is time- and height-varying.

| Parameter | Value |
| --- | --- |
| Domain | 5000 m x 5000 m x 2000 m |
| Grid (reference run) | $200^3$ |
| Simulation time | 24 h (0900 LST Day 33 - 0900 LST Day 34) |
| Time step | $dt = 0.2$ s |
| Surface BC | observed screen temperature at $z_T = 1.2$ m (`optSurfBC = 2`) |
| Geostrophic wind | time- and height-varying (`optGeoWind = 1`) |
| Coriolis parameter | $f = -8.26 \times 10^{-5}$ s$^{-1}$ (lat ~34$^\circ$ S) |
| Reference temperature | $T_0 = 278.5$ K |
| Roughness length | $z_{0m} = z_{0T} = 0.01$ m |
| Inversion strength | 0.001 K m$^{-1}$ |
| SGS model | LASDD-SM (`optSgs = 1`) |
| Damping layer | above $z = 1500$ m |

The reference run shown below is `examples/DC_Wangara/runs/128x128x128_LASDD_SM_SP`.

**JAX-ALFA Configuration**

In [ ]:
# ============================================================
# Imports
# ============================================================

import numpy as np


# ============================================================
# User Input
# ============================================================

# ------------------------------------------------------------
# Platform options
# ------------------------------------------------------------
use_double_precision = False
# 0: use CPU, 1: use GPU
optGPU = 1
GPU_ID = 0

# ------------------------------------------------------------
# Domain configuration
# ------------------------------------------------------------

# Domain size (m)
l_x = 5000
l_y = 5000
l_z = 2000

# Number of grid points
nx = 200
ny = 200
nz = 200

# ------------------------------------------------------------
# Time integration configuration
# ------------------------------------------------------------

# Change this if it is a restart run
istep = 1

# Time stepping and simulation time
dt = 0.2          # unit: sec
SimTime = 86400   # 24 hours, unit: sec

# Galilean transformation (m/s)
Ugal = 0

# ------------------------------------------------------------
# Surface configuration
# ------------------------------------------------------------

# optSurfFlux: 0 = homogeneous, 1 = heterogeneous
optSurfFlux = 0

# optSurfBC: 0 = constant flux
#            1 = time-varying flux (from SurfaceBCFile)
#            2 = time-varying surface temperature (from SurfaceBCFile)
optSurfBC = 2

# Roughness lengths (m)
z0m = 0.01
z0T = 0.01   # not used when zTemperature > 0; kept for backward compatibility

# Screen-level temperature reference height (m).
# The Wangara surface temperature is measured at 1.2 m above the ground.
# MOST heat-flux denominator uses log(z1/zTemperature) + psi_h - psi_h0.
zTemperature = 1.2

# SensibleHeatFlux: not used when optSurfBC >= 1; set to 0 as placeholder
SensibleHeatFlux = 0.0  # K m/s

# Path to surface BC file (relative to run directory)
SurfaceBCFile = 'input/SurfaceBC.npz'

# ------------------------------------------------------------
# Forcing configuration
# ------------------------------------------------------------

# Geostrophic wind option:
#   0 = constant Ug2, Vg2 from Config
#   1 = time + height varying, loaded from GeoWindFile
optGeoWind = 1

# Constant geostrophic wind (m/s) — used only when optGeoWind = 0
Ug2 = -5.34
Vg2 = -0.43

# Path to geostrophic wind file (relative to run directory)
GeoWindFile = 'input/GeoWind.npz'

# Coriolis parameter: f = 2*Omega*sin(lat)
# Wangara, Australia (~34 deg S)  =>  f < 0
f_coriolis = -8.26e-5  # unit: 1/s

# Temperature inversion
inversion = 0.001

# Buoyancy calculation: 0 = use reference T_0, 1 = use local THv
optBuoyancy = 1 

# Reference temperature (K) — screen-level potential temperature at t=0
# theta_screen = (5.3 + 273.16) + 1.2*10/1000 = 278.47 K  (from Wangara_Sfc3309.txt)
T_0 = 278.5

# ------------------------------------------------------------
# Subgrid-scale configuration
# ------------------------------------------------------------

# SGS model: 0 = Static SM, 1 = LASDD-SM, 2 = LASDD-WL, 3 = LAD-SM, 4 = LAD-WL
optSgs = 1

# Dynamic SGS update frequency (every N steps)
dynamicSGS_call_time = 1

# Filter to grid ratio (FGR=1: implicit filtering with dealiasing)
FGR = 2

# Initial SGS coefficients (used before first dynamic update when dynamicSGS_call_time > 1)
Cs2 = 0.1 ** 2        # SM models: initial Cs^2
Cwl = 0.1 ** 2        # WL models: initial C_WL
Cs2PrRatio = Cs2 / 1.0
CwlPrRatio = Cwl / 1.0

# ------------------------------------------------------------
# Damping layer configuration
# ------------------------------------------------------------

optDamping = 1       # 1: activate Rayleigh damping
z_damping  = 1500.0  # unit: m
RelaxTime  = 300.0   # unit: s

# ------------------------------------------------------------
# Statistics computation
# ------------------------------------------------------------

SampleInterval_sec   = 10.0       # collect a sample every 10 s
OutputInterval_sec   = 60.0       # output averaged stats every 60 s
Output3DInterval_sec = SimTime  # output 3D fields only at the end of the run

# ------------------------------------------------------------
# Large-scale advection forcing
# ------------------------------------------------------------
# 0: none, 1: time/height-varying (from AdvectionFile)
optAdvection = 0
AdvectionFile = 'input/AdvForcing.npz'

# ------------------------------------------------------------
# Moisture configuration
# ------------------------------------------------------------
# 0: dry run, 1: prognostic specific humidity Q
optMoisture = 0
# Screen-level moisture reference height (m); 0 = use z0T
zMoisture = 0.0
# Surface moisture flux (kg/kg m/s); used when optMoistureSurfBC = 0
MoistureFlux = 0.0
# 0: constant flux, 1: time-varying flux, 2: time-varying surface Q
optMoistureSurfBC = 0
MoistureSurfaceBCFile = 'input/MoistureSurfaceBC.npz'
# Specific humidity lapse rate above domain top (kg/kg/m); 0 = zero gradient
q_inversion = 0.0